# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/25RA/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

### The Data Contract in Plain Words (Five Answers)

1. **What one row means (Unit of Analysis):**  
   One row = **One unique published content item (`content_id`)** within an anonymized client tenant (`client_id`).
2. **Which table(s) we will use:**  
   The core content performance table (`content_refresh_anonymized.csv` / `dim_content` joined to aggregated `fact_content_daily_performance` partitions via DuckDB).
3. **Which time window:**  
   - **Feature Observation Window:** Trailing 90 calendar days of search impressions, clicks, average position, and engagement.
   - **Outcome Window:** 30-day evaluation window comparing `impressions_last_30d` against `impressions_prev_30d` to determine whether trend is declining.
4. **What we predict or rank (Label / Proxy):**  
   `is_declining_label` (Binary: 1 if `trend_direction == 'down'`, else 0), scored via predicted decline probability $\hat{P}(\text{decline} = 1 \mid X)$ to generate a ranked review queue.
5. **One thing we deliberately exclude:**  
   `trend_pct` (and `*_last_30d` / `*_prev_30d` windowed metrics). `trend_pct` is the exact percentage change the label is calculated from; including it constitutes fatal data leakage.

In [1]:
import os, sys
import duckdb
import pandas as pd
import numpy as np

# Safe path resolution
csv_candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv"
]
csv_path = next(p for p in csv_candidates if os.path.exists(p)).replace("\\", "/")

con = duckdb.connect()
print("DuckDB engine initialized.")
print(f"Contract Source Table: {csv_path}")


DuckDB engine initialized.
Contract Source Table: data/raw/content_refresh_anonymized.csv


## 2. Fields: feature / label / context / excluded

Every field in the dataset is assigned to one of four strict contract buckets:

1. **Features (Pre-decision inputs):**  
   `impressions_90d`, `clicks_90d`, `avg_position`, `position_tier`, `ctr`, `engagement_rate`, `scroll_rate`, `word_count`, `content_age_days`, `days_since_last_update`, `content_type`, `main_intent`, `search_volume`, etc. (Knowable before the review decision).
2. **Label / Target (Ground truth outcome):**  
   `trend_direction`, `trend_pct`, `is_declining_label`.
3. **Context (Identifiers for grouping & holdout splits):**  
   `content_id` (page key), `client_id` (cluster split key).
4. **Excluded (Quarantined to prevent leakage):**  
   `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`, and `trend_pct` (Direct derivation components of the label).

In [2]:
# Verifying column classification contract via DuckDB
contract_summary = con.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT client_id) as total_clients,
        COUNT(DISTINCT content_id) as total_content_ids,
        COUNT(DISTINCT content_type) as content_types_count
    FROM '{csv_path}'
""").df()

print("Dataset Summary:")
print(contract_summary.to_string(index=False))


Dataset Summary:
 total_rows  total_clients  total_content_ids  content_types_count
      30000             32              30000                    3


## 3. Verify it with queries (grain, counts, missing values, windows)

### Part A: Exactly Three Verification Queries

1. **Query 1: The Grain** — Proving that one row really is one unique content item (`content_id`).
2. **Query 2: Slice Row Count & Date/Age Span** — Measuring catalog row count, content age span, and days since update.
3. **Query 3: Availability (checked with `IS TRUE`)** — Measuring how many rows survive an active visibility filter (`impressions_90d >= 100 AND days_with_impressions >= 10`).

### Part B: The Five Features Frame
Five core features with an "available when?" justification for each.

### Part C: The Deliberate Leakage Trap Experiment
Demonstrating the trap: adding `trend_pct` on purpose, watching the score jump artificially to 1.000, then deleting it to keep the honest score.

In [3]:
# ==============================================================================
# Query 1: The Grain (Zero duplicates on content_id)
# ==============================================================================
grain_check = con.sql(f"""
    SELECT content_id, COUNT(*) as c
    FROM '{csv_path}'
    GROUP BY content_id
    HAVING c > 1
""").df()
print(f"Query 1: Grain verification duplicates count = {len(grain_check)} (Grain holds: 1 row = 1 unique content_id)")

# ==============================================================================
# Query 2: Slice Row Count and Age/Span
# ==============================================================================
span_check = con.sql(f"""
    SELECT 
        COUNT(*) as slice_row_count,
        MIN(content_age_days) as min_age_days,
        MAX(content_age_days) as max_age_days,
        MIN(days_since_last_update) as min_update_days,
        MAX(days_since_last_update) as max_update_days
    FROM '{csv_path}'
""").df()
print("\nQuery 2: Slice Row Count & Span:")
print(span_check.to_string(index=False))

# ==============================================================================
# Query 3: Availability Filter (checked with IS TRUE)
# ==============================================================================
avail_check = con.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(CASE WHEN (impressions_90d >= 100 AND days_with_impressions >= 10) IS TRUE THEN 1 END) as active_surviving_rows,
        ROUND(COUNT(CASE WHEN (impressions_90d >= 100 AND days_with_impressions >= 10) IS TRUE THEN 1 END) * 100.0 / COUNT(*), 1) as pct_surviving
    FROM '{csv_path}'
""").df()
print("\nQuery 3: Availability (checked with IS TRUE):")
print(avail_check.to_string(index=False))

# ==============================================================================
# Part B: Build Five Features Frame
# ==============================================================================
# Five core features:
# 1. content_age_days      -> knowable at the decision moment because publication date is logged in CMS.
# 2. days_since_last_update -> knowable at the decision moment because last editorial edit timestamp is recorded.
# 3. impressions_90d       -> knowable at the decision moment because Search Console logs prior 90-day search visibility.
# 4. avg_position          -> knowable at the decision moment because historical ranking is computed over the completed 90-day window.
# 5. ctr                   -> knowable at the decision moment because clicks/impressions are historical user behaviors.

feature_cols = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr']
df_features = con.sql(f"""
    SELECT 
        content_id, client_id,
        content_age_days, days_since_last_update, impressions_90d, avg_position, ctr,
        CASE WHEN LOWER(trend_direction) = 'down' THEN 1 ELSE 0 END as is_declining_label,
        trend_pct
    FROM '{csv_path}'
""").df()

print(f"\nFive-Feature Frame built: {len(df_features):,} rows × {len(feature_cols)} features.")
print(df_features[['content_id'] + feature_cols].head(3).to_string(index=False))

# ==============================================================================
# Part C: The Deliberate Leakage Trap Experiment
# ==============================================================================
from sklearn.tree import DecisionTreeClassifier

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

X_honest = df_features[feature_cols].fillna(0)
y = df_features['is_declining_label'].values

# 1. Honest model
honest_tree = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42)
honest_tree.fit(X_honest, y)
honest_p50 = precision_at_k(honest_tree.predict_proba(X_honest)[:, 1], y, 50)
print(f"\n1. Honest Model Precision@50: {honest_p50:.3f}")

# 2. Spring the trap: add ONE label-derived column on purpose (trend_pct)
X_leaky = df_features[feature_cols + ['trend_pct']].fillna(0)
leaky_tree = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42)
leaky_tree.fit(X_leaky, y)
leaky_p50 = precision_at_k(leaky_tree.predict_proba(X_leaky)[:, 1], y, 50)
print(f"2. Trap Model (with leaky trend_pct) Precision@50: {leaky_p50:.3f}  <- Suspiciously perfect!")

# 3. Delete the leaky column and confirm honest model is kept
del X_leaky
print(f"3. Leaky column deleted. Honest Precision@50 retained: {honest_p50:.3f}")


Query 1: Grain verification duplicates count = 0 (Grain holds: 1 row = 1 unique content_id)

Query 2: Slice Row Count & Span:
 slice_row_count  min_age_days  max_age_days  min_update_days  max_update_days
           30000            90           564                1              373

Query 3: Availability (checked with IS TRUE):
 total_rows  active_surviving_rows  pct_surviving
      30000                  22003           73.3

Five-Feature Frame built: 30,000 rows × 5 features.
          content_id  content_age_days  days_since_last_update  impressions_90d  avg_position  ctr
content_304f48230142               187                      20             3803          10.6 0.76
content_a1fb4e703a9e               445                      25            15320          20.3 0.05
content_9aa793d4d895               141                      20            12581          36.5 0.09

1. Honest Model Precision@50: 0.720
2. Trap Model (with leaky trend_pct) Precision@50: 1.000  <- Suspiciously perfect!


## 4. Data limits

**Named Limitations of This Data Slice:**

1. **Absence of External SERP Competitor Actions:**  
   The dataset captures page-level search outcomes (impressions, clicks, average position), but does not record external competitor movements (e.g., a competitor publishing a comprehensive 5,000-word guide or Google rolling out an AI Overview for the query). Therefore, a traffic drop indicates *that* visibility eroded, but cannot isolate whether the cause was internal content obsolescence versus external competitive displacement.
2. **Sentinel Value for Unranked Pages (`avg_position == 0`):**  
   A value of 0 in Google Search Console denotes "no ranking data / unranked", not position zero. Failing to treat 0 as an unranked sentinel would severely distort position-drift calculations.
3. **Unbalanced Tracking Histories:**  
   Different client tenants onboarded at different calendar times; comparing raw aggregate counts across clients without per-client normalization introduces tenant-size bias.

In [4]:
# Demonstrating named limitations in the slice
unranked_sentinels = (df_features['avg_position'] == 0).sum()
print("Demonstrating Data Limits in the Slice:")
print(f"1. avg_position == 0 sentinels: {unranked_sentinels:,} pages ({unranked_sentinels/len(df_features):.1%})")
print("   -> Handled: Filtered or flagged as unranked; never treated as rank zero.")
print("2. Observational limit:")
print("   -> External SERP feature changes (AI Overviews, featured snippets) are unobserved in this slice.")


Demonstrating Data Limits in the Slice:
1. avg_position == 0 sentinels: 1,205 pages (4.0%)
   -> Handled: Filtered or flagged as unranked; never treated as rank zero.
2. Observational limit:
   -> External SERP feature changes (AI Overviews, featured snippets) are unobserved in this slice.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.